In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

working_dir = Path.cwd().resolve()
BASE_DIR = next(
    (candidate for candidate in (working_dir, *working_dir.parents)
     if (candidate / "ml" / "data" / "raw").exists()),
    working_dir,
)
RAW_DIR = BASE_DIR / "ml" / "data" / "raw"
OUTPUT_DIR = BASE_DIR / "ml" / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

files = {
    "perishable_inventory": "perishable_inventory.csv",
    "inventory_batches": "inventory_batches.csv",
    "discount_scenarios": "discount_scenarios_simulated.csv",
    "calendar": "calendar.csv",
    "sell_prices": "sell_prices.csv",
    "products": "products.csv"
}

dataset_report = []
missing_report = []

for name, filename in files.items():
    file_path = RAW_DIR / filename

    if not file_path.exists():
        print(f"\nFile not found: {file_path}")
        continue

    df = pd.read_csv(file_path)

    print("\n" + "=" * 60)
    print("DATASET:", name)
    print("=" * 60)

    print("\nShape:", df.shape)
    print("\nFirst five rows:")
    print(df.head())

    print("\nColumn names:")
    print(df.columns.tolist())

    print("\nData types:")
    print(df.dtypes)

    print("\nMissing values:")
    print(df.isnull().sum())

    print("\nDuplicate rows:", df.duplicated().sum())

    print("\nSummary statistics:")
    print(df.describe(include="all").T)

    for column in df.columns:
        missing_report.append({
            "Dataset": name,
            "Column": column,
            "Missing_Count": int(df[column].isna().sum()),
            "Missing_Percentage": round(
                df[column].isna().mean() * 100, 2
            )
        })

    dataset_report.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing_Values": int(df.isna().sum().sum()),
        "Duplicate_Rows": int(df.duplicated().sum())
    })

    if name == "perishable_inventory":
        if "date" in df.columns:
            df["date"] = pd.to_datetime(
                df["date"], errors="coerce"
            )
            print("\nDate range:")
            print("Start:", df["date"].min())
            print("End:", df["date"].max())

        if "units_sold" in df.columns:
            print("\nTotal units sold:", df["units_sold"].sum())
            print("Average units sold:", df["units_sold"].mean())

        if "waste_units" in df.columns:
            print("\nTotal waste units:", df["waste_units"].sum())
            print("Average waste units:", df["waste_units"].mean())

        if {"product_id", "waste_units"}.issubset(df.columns):
            print("\nWaste by product:")
            print(
                df.groupby("product_id")["waste_units"]
                .sum()
                .sort_values(ascending=False)
            )

if dataset_report:
    report_df = pd.DataFrame(dataset_report)
    report_df.to_csv(
        OUTPUT_DIR / "data_understanding_report.csv",
        index=False
    )

    missing_df = pd.DataFrame(missing_report)
    missing_df.to_csv(
        OUTPUT_DIR / "missing_values_report.csv",
        index=False
    )

    print("\n" + "=" * 60)
    print("FINAL DATASET REPORT")
    print("=" * 60)
    print(report_df.to_string(index=False))

    print("\nReports saved in:", OUTPUT_DIR)
    print("Data understanding completed successfully.")
else:
    print("\nNo datasets were loaded. Check the raw data folder.")


DATASET: perishable_inventory

Shape: (1800, 12)

First five rows:
   product_id        date category  opening_stock  units_received  units_sold  \
0        1001  2026-04-01    Dairy             27               0           6   
1        1001  2026-04-02    Dairy             30              12           5   
2        1001  2026-04-03    Dairy             38               0          14   
3        1001  2026-04-04    Dairy             49               0          11   
4        1001  2026-04-05    Dairy             19              13          12   

   unit_price  discount_pct  days_to_expiry  temperature_c  waste_units  \
0        32.0           0.0               1            4.1            0   
1        32.0           0.0               5            3.7            0   
2        32.0           0.0               2            4.6            0   
3        32.0           0.0               0            5.7            0   
4        32.0           0.0               0            3.0            


DATASET: sell_prices

Shape: (6841121, 4)

First five rows:
  store_id        item_id  wm_yr_wk  sell_price
0     CA_1  HOBBIES_1_001     11325        9.58
1     CA_1  HOBBIES_1_001     11326        9.58
2     CA_1  HOBBIES_1_001     11327        8.26
3     CA_1  HOBBIES_1_001     11328        8.26
4     CA_1  HOBBIES_1_001     11329        8.26

Column names:
['store_id', 'item_id', 'wm_yr_wk', 'sell_price']

Data types:
store_id       object
item_id        object
wm_yr_wk        int64
sell_price    float64
dtype: object

Missing values:
store_id      0
item_id       0
wm_yr_wk      0
sell_price    0
dtype: int64



Duplicate rows: 0

Summary statistics:


                count unique          top    freq          mean         std  \
store_id      6841121     10         TX_2  701214           NaN         NaN   
item_id       6841121   3049  FOODS_3_587    2820           NaN         NaN   
wm_yr_wk    6841121.0    NaN          NaN     NaN  11382.943423  148.610026   
sell_price  6841121.0    NaN          NaN     NaN      4.410952    3.408814   

                min      25%      50%      75%      max  
store_id        NaN      NaN      NaN      NaN      NaN  
item_id         NaN      NaN      NaN      NaN      NaN  
wm_yr_wk    11101.0  11247.0  11411.0  11517.0  11621.0  
sell_price     0.01     2.18     3.47     5.84   107.32  



DATASET: products

Shape: (10, 6)

First five rows:
   product_id  product_name category  shelf_life_days  selling_price  \
0        1001    Milk 500ml    Dairy                5           32.0   
1        1002    Yogurt Cup    Dairy               10           25.0   
2        1003    Bread Loaf   Bakery                4           40.0   
3        1004   Paneer 200g    Dairy                7           85.0   
4        1005  Spinach Pack  Produce                3           30.0   

   unit_cost  
0       24.0  
1       17.0  
2       26.0  
3       62.0  
4       18.0  

Column names:
['product_id', 'product_name', 'category', 'shelf_life_days', 'selling_price', 'unit_cost']

Data types:
product_id           int64
product_name        object
category            object
shelf_life_days      int64
selling_price      float64
unit_cost          float64
dtype: object

Missing values:
product_id         0
product_name       0
category           0
shelf_life_days    0
selling_price      0
unit_c